# LangGraph G2 — The model as a node
CampusAI v1 replaces keyword matching with a language model. Three facts about models decide
the whole design of an agent:

- A model is **text in, text out**. It cannot run code, read files or call APIs.
- A model has **no memory**. It sees only what is in the current request.
- A model reads a **context**: a list of messages with roles (system instructions, user turns,
  its own earlier replies, and later tool results). Every token in it costs money and attention.

In LangGraph a model call is *just a node*: a function that reads the conversation from the
state and returns the reply.

```text
           +-----------+
START ---> |  chatbot  | ---> END        state = {"messages": [...]}
           +-----------+
```

One new idea makes this work: a **reducer**. By default a node's update *replaces* a key. For a
conversation we want updates to *append*. Declaring `messages: Annotated[list, add_messages]`
tells LangGraph how to merge every update into that key. Get this idea now; it returns in G10
when several nodes write to the same key at once.

### Step 1 — Reducers: replace versus append

Two identical nodes each return one message. With a plain list key the second replaces the
first; with the `add_messages` reducer both are kept.

> **Why LangGraph has a *reducer***
>
> Several nodes, and sometimes several nodes at once, write to the same key. LangGraph refuses to guess whether the new value should replace, append to or merge with the old one, so you declare it once on the key. A conversation is the first place this matters; parallel branches (G10) are the second.

In [ ]:
from langgraph.graph.message import add_messages       # LangGraph: reducer that appends messages (and de-duplicates by id)

class PlainState(TypedDict):                           # ours: no reducer -> updates replace the key
    messages: list

class ChatState(TypedDict):                            # ours: reducer -> updates append to the key
    messages: Annotated[list, add_messages]

def say_a(state): return {"messages": [AIMessage("A")]}   # ours: two tiny nodes (AIMessage is LangChain)
def say_b(state): return {"messages": [AIMessage("B")]}

for schema, label in [(PlainState, "plain list  "), (ChatState, "add_messages")]:
    g = StateGraph(schema)
    g.add_node("say_a", say_a); g.add_node("say_b", say_b)
    g.add_edge(START, "say_a"); g.add_edge("say_a", "say_b"); g.add_edge("say_b", END)
    out = g.compile().invoke({"messages": [HumanMessage("start")]})
    print(f"{label} -> {[text_of(m) for m in out['messages']]}")

> **What just happened**
>
> Two identical graphs: START -> say_a -> say_b -> END. With the plain list, `say_b`'s update `{"messages": [B]}` replaced what `say_a` had written, so only `['B']` survived. With `add_messages` the same update was appended, so the final list holds the start message, A and B. The nodes did not change at all; only the reducer on the key did.

### Step 2 — The chatbot graph

The node calls the model with a **system prompt** (standing instructions: who the assistant is,
how to behave) plus the conversation so far, and returns the reply as a one-message update.
The reply carries `usage_metadata`: the tokens you paid for on this call.

In [ ]:
CAMPUS_PERSONA = "You are CampusAI, the helpdesk assistant of Northfield University. Be concise and friendly."   # ours

def chatbot(state: ChatState):                                       # ours: the model as a node
    reply = model.invoke([SystemMessage(CAMPUS_PERSONA)] + state["messages"])   # LangChain: one request, one AIMessage
    return {"messages": [reply]}                                      # the reducer appends it

g = StateGraph(ChatState)
g.add_node("chatbot", chatbot)
g.add_edge(START, "chatbot")
g.add_edge("chatbot", END)
campusai_v1 = g.compile()

out = campusai_v1.invoke({"messages": [HumanMessage("Hi! What can you do?")]})   # LangGraph: run the graph
show_messages(out["messages"])
print("usage:", out["messages"][-1].usage_metadata)                # LangChain: token counts on the AIMessage

> **What just happened**
>
> START -> `chatbot` -> END: one node, one model call. The node sent the system prompt plus the single human message, received one AIMessage and returned it; the reducer appended it, which is why the printed state shows two messages. The usage line is what that one call cost in tokens.

### Step 3 — The graph forgets between runs

Two separate invocations are two separate states. The model only ever sees what is in the
current `messages`, so the second run cannot know the name given in the first. The message list
*is* the model's memory; G5 makes LangGraph carry it between runs.

In [ ]:
first = campusai_v1.invoke({"messages": [HumanMessage("My name is Rahul.")]})   # LangGraph invoke, LangChain message
print("run 1 :", text_of(first["messages"][-1]))

second = campusai_v1.invoke({"messages": [HumanMessage("What is my name?")]})     # a brand-new state
print("run 2 :", text_of(second["messages"][-1]))

third = campusai_v1.invoke({"messages": first["messages"] + [HumanMessage("What is my name?")]})   # we carry the history by hand
print("run 3 :", text_of(third["messages"][-1]), "  <- only because WE passed the earlier messages in")

> **What just happened**
>
> Three separate runs, three separate states. Run 1's state held the name and the reply. Run 2 started from a fresh list containing only the question, so the model could not know the name. Run 3 worked only because we passed run 1's messages back in by hand: the graph itself kept nothing. G5 makes LangGraph do that carrying for us.

### Recap

- **The problem we started with:** keyword rules cannot understand language; and a model has no memory of its own.
- **What we added:** a model node over a `messages` key with the `add_messages` reducer, and a system prompt.
- **What you saw in the output:** the reducer demo kept both messages; run 2 forgot the name and run 3 knew it only because we re-sent it.
- **Carry forward:** G3 keeps the chatbot node and adds a second node for tools, plus an edge that points back, which is all an agent loop is.